# Getting Started with Automated-LLM-Probes

In [1]:
from __future__ import annotations
import os
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [2]:
models = alp.ready_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name']:20s} {m['api']:12s} {m['model_id']}")

|Models available: 56|

  GPT-3.5-Turbo        openai       gpt-3.5-turbo
  Moonshot-v1-8k       moonshot     moonshot-v1-8k
  Moonshot-v1-128k     moonshot     moonshot-v1-128k
  GPT-4o               openai       gpt-4o-2024-08-06
  Qwen-Turbo           qwen         qwen-turbo
  o4-mini              openai       o4-mini-2025-04-16
  GPT-4.1              openai       gpt-4.1-2025-04-14
  GPT-4.1-mini         openai       gpt-4.1-mini-2025-04-14
  GPT-4.1-nano         openai       gpt-4.1-nano-2025-04-14
  Kimi-K2              moonshot     moonshot-v1-32k
  Qwen3-235B-Instruct  qwen         qwen3-235b-a22b-instruct-2507
  GPT-5                openai       gpt-5-2025-08-07
  GPT-5-mini           openai       gpt-5-mini-2025-08-07
  Qwen-Max             qwen         qwen-max
  Qwen-Plus            qwen         qwen-plus
  Claude Sonnet 4.5    claude       claude-sonnet-4-5-20250929
  MiniMax-M2.5         openrouter   minimax/minimax-m2.5
  Claude Haiku 4.5     claude       claude-haiku-4-

### Probed tasks

In [6]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if (d.is_dir()) and ('.' not in d.name):
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

|Probed tasks: 4|

  1. AUT     :  6893
  2. CWT     :  6715
  3. DAT     :  6374


### Tests availabe

In [7]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  CWT : Creative Writing Task
  DAT : Divergent Association Task


### Tiny trial-run

In [14]:
models = [m for m in alp.ready_models() if m["name"] == "Llama-4 Maverick"]
alp.collect("DAT", models=models, n_per_model=350)

  Llama-4 Maverick: 320 collected, 30 to collect


DAT: 100%|███████████████████████████████████████████████████████████████| 30/30 [00:53<00:00,  1.79s/it]


### Load functions

In [15]:
from __future__ import annotations
import re, pandas as pd
import glove_word_embeddings as gwe

def parse_dat(raw):
    text = str(raw or "").strip().strip('"').strip("'")
    tokens = re.split(r"[,\n\r]+", text)
    nouns = [n for n in (gwe.pre.clean_word(t) for t in tokens) if n][:10]
    return nouns + [""] * (10 - len(nouns))

def parse_aut(raw):
    uses = []
    for line in re.split(r"[\n\r]+", str(raw or "")):
        line = re.sub(r"^\s*[\d\.\)\-]+\s*", "", line)
        if toks := [t for t in (gwe.pre.clean_word(t) for t in line.split()) if t]:
            uses.append(" ".join(toks))
    return ", ".join(uses)

def parse_cwt(raw):
    text = re.sub(r"^#+\s*.*$", "", str(raw or ""), flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def load_task(task: str) -> pd.DataFrame:
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.parse_and_merge(task),orient='index')
    if task == "dat":
        parsed = df["raw"].map(parse_dat)
        df[[f"noun_{i}" for i in range(10)]] = parsed.tolist()
        df["response_clean"] = parsed.map(lambda x: ", ".join(n for n in x if n))
        extra = [f"noun_{i}" for i in range(10)]
    elif task == "aut":
        df["object"] = df["prompt"].str.extract(
            r"object: (.+?)\?", expand=False).str.strip()
        df["response_clean"] = df["raw"].map(parse_aut)
        extra = ["object"]
    elif task == "cwt":
        cues = df["prompt"].str.extract(
            r"words: (.+?)\.", expand=False).str.strip().str.split(r",\s*")
        df[["cue_0", "cue_1", "cue_2"]] = pd.DataFrame(cues.tolist()).iloc[:, :3]
        df["response_clean"] = df["raw"].map(parse_cwt)
        extra = ["cue_0", "cue_1", "cue_2"]
    else:
        raise ValueError(f"Unknown task: {task}")
    cols = ["task", "model_name", "model_id", 
            "provider", "rep", "temperature_std"] + extra + [
        "prompt", "response_clean", "ts_utc", "hash"]
    return df[[c for c in cols if c in df.columns]].sort_values(
        ["model_name", "rep"]).reset_index(drop=True)
        
print('All functions loaded...')

All functions loaded...


### Parse & merge data

In [16]:
for task in (
    "dat", 
#     "aut",
#     "cwt"
):
    print(f"Parsing {task.upper()}...")
    df = load_task(task)
    print(df.shape)
    df.to_csv(f"./data/{task}.csv", index=False)

Parsing DAT...


dat:   0%|▏                                                          | 15/6424 [00:25<2:59:02,  1.68s/it]


KeyboardInterrupt: 